In [26]:
import hdbscan
import pandas as pd
import sqlalchemy as db
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download

In [2]:
dw_local_path = hf_hub_download(
    repo_id="astaileyyoung/CineFaceDB",
    filename="CineFaceDW_agg.duckdb",
    repo_type="dataset",
    local_dir="."
)

In [3]:
engine = db.create_engine("duckdb:///CineFaceDW_agg.duckdb")

In [7]:
g = pd.read_sql("SELECT * FROM vwDirectorStats WHERE cnt > 5", engine)
g

,person_id,name,cnt,avg_f_per_fr,z_size_mv_mean,z_size_mv_g_mean,z_size_mv_std,z_size_mv_g_std,z_top1_mv_mean,z_top1_mv_g_mean,...,z_v_density_mv_mean,z_v_density_mv_g_mean,z_pct_top1_mv_mean,z_pct_top1_mv_g_mean,z_v_pct_top1_mv_mean,z_v_pct_top1_mv_g_mean,z_pct_face_mv_mean,z_pct_face_mv_g_mean,z_v_pct_face_mv_mean,z_v_pct_face_mv_g_mean
0,1,George Lucas,6,1.597648,-0.053352,0.814914,0.783111,0.766601,-0.190065,0.802465,...,0.000000,0.000000,0.277344,0.991328,-0.111087,-0.397079,-0.226912,0.660440,-0.229422,0.845807
1,40,Orson Welles,9,1.770517,0.697528,0.110828,0.808529,0.445027,0.889053,0.204514,...,0.745900,0.449870,0.888112,0.327788,0.070874,0.214835,0.630211,0.140793,1.023617,0.451762
2,68,Fritz Lang,33,1.873031,-0.432220,-0.640721,0.738070,0.369027,-0.447708,-0.691414,...,0.064489,-0.170618,0.384600,-0.147534,0.320935,0.367024,-0.660249,-0.775029,-0.107610,-0.558840
3,138,Quentin Tarantino,7,1.581424,-0.217380,0.863364,0.969357,1.334367,-0.287119,0.842447,...,-0.286160,0.325392,0.116981,1.032773,-0.151320,-0.850129,-0.378893,0.631704,0.342752,1.289849
4,190,Clint Eastwood,32,2.017244,-0.416212,0.314150,0.702603,0.667530,-0.269060,0.586161,...,0.029670,0.063425,-0.578839,0.328151,0.495116,0.198262,-0.136572,0.651149,-0.389156,0.490620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,940663,Chester M. Franklin,8,1.618405,0.397221,-0.568999,0.979921,0.185540,0.344414,-0.740273,...,-0.017444,0.621329,-0.074203,0.162069,0.396381,0.419476,0.359364,-0.866964,-0.004262,-0.903669
329,1021685,E. Mason Hopper,6,1.722700,-0.272303,-0.544496,0.875573,0.382297,-0.368806,-0.691033,...,-0.234674,-0.300615,-0.142999,-0.409118,0.188546,0.391993,-0.542793,-0.771631,-0.404528,-0.799606
330,1036754,Reginald Barker,7,1.812715,0.462617,-0.635704,0.387553,0.375400,0.511501,-0.746425,...,0.359743,0.539909,-0.135964,-0.138941,0.476722,0.568228,0.635482,-0.770480,0.400587,-0.831591
331,1065336,Melville W. Brown,6,1.830765,-0.722426,-0.799145,0.455469,0.263664,-0.791800,-0.919072,...,-0.030822,-0.239791,-0.170180,-0.596750,0.235600,0.280355,-0.923024,-0.982991,-0.877001,-1.104247


In [10]:
features = ["z_size_mv_g_mean", "z_f_per_fr_mv_g_mean", "z_gini_mv_g_mean", "z_dist_mv_g_mean", "z_vert_mv_g_mean", "z_h_spread_mv_g_mean"]
X = g[features]
names = g['name']

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
g['cluster'] = kmeans.fit_predict(X)
g

,person_id,name,cnt,avg_f_per_fr,z_size_mv_mean,z_size_mv_g_mean,z_size_mv_std,z_size_mv_g_std,z_top1_mv_mean,z_top1_mv_g_mean,...,z_v_density_mv_g_mean,z_pct_top1_mv_mean,z_pct_top1_mv_g_mean,z_v_pct_top1_mv_mean,z_v_pct_top1_mv_g_mean,z_pct_face_mv_mean,z_pct_face_mv_g_mean,z_v_pct_face_mv_mean,z_v_pct_face_mv_g_mean,cluster
0,1,George Lucas,6,1.597648,-0.053352,0.814914,0.783111,0.766601,-0.190065,0.802465,...,0.000000,0.277344,0.991328,-0.111087,-0.397079,-0.226912,0.660440,-0.229422,0.845807,0
1,40,Orson Welles,9,1.770517,0.697528,0.110828,0.808529,0.445027,0.889053,0.204514,...,0.449870,0.888112,0.327788,0.070874,0.214835,0.630211,0.140793,1.023617,0.451762,4
2,68,Fritz Lang,33,1.873031,-0.432220,-0.640721,0.738070,0.369027,-0.447708,-0.691414,...,-0.170618,0.384600,-0.147534,0.320935,0.367024,-0.660249,-0.775029,-0.107610,-0.558840,3
3,138,Quentin Tarantino,7,1.581424,-0.217380,0.863364,0.969357,1.334367,-0.287119,0.842447,...,0.325392,0.116981,1.032773,-0.151320,-0.850129,-0.378893,0.631704,0.342752,1.289849,0
4,190,Clint Eastwood,32,2.017244,-0.416212,0.314150,0.702603,0.667530,-0.269060,0.586161,...,0.063425,-0.578839,0.328151,0.495116,0.198262,-0.136572,0.651149,-0.389156,0.490620,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,940663,Chester M. Franklin,8,1.618405,0.397221,-0.568999,0.979921,0.185540,0.344414,-0.740273,...,0.621329,-0.074203,0.162069,0.396381,0.419476,0.359364,-0.866964,-0.004262,-0.903669,4
329,1021685,E. Mason Hopper,6,1.722700,-0.272303,-0.544496,0.875573,0.382297,-0.368806,-0.691033,...,-0.300615,-0.142999,-0.409118,0.188546,0.391993,-0.542793,-0.771631,-0.404528,-0.799606,4
330,1036754,Reginald Barker,7,1.812715,0.462617,-0.635704,0.387553,0.375400,0.511501,-0.746425,...,0.539909,-0.135964,-0.138941,0.476722,0.568228,0.635482,-0.770480,0.400587,-0.831591,3
331,1065336,Melville W. Brown,6,1.830765,-0.722426,-0.799145,0.455469,0.263664,-0.791800,-0.919072,...,-0.239791,-0.170180,-0.596750,0.235600,0.280355,-0.923024,-0.982991,-0.877001,-1.104247,4


In [12]:
pca = PCA(n_components=2)
components = pca.fit_transform(X)

In [21]:
pca_df = pd.DataFrame(data=components, columns=['PC1', 'PC2'], index=g.index)
pca_df['cluster'] = g['cluster'].astype(str)
pca_df['name'] = names

In [22]:
fig = px.scatter(
    pca_df.reset_index(), 
    x='PC1', y='PC2', 
    hover_name="name",
    color='cluster',
    title="Cinematographic Style Clusters (PCA)"
)
fig.show()

In [23]:
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X) 

# 2. Dimensionality Reduction (For Visualization ONLY)
pca = PCA(n_components=2)
components = pca.fit_transform(X)

# 3. Create the Visualization DataFrame
# We combine the 2D coordinates with the names and the clusters
vis_df = pd.DataFrame(
    data = components, 
    columns = ['PC1', 'PC2']
)
vis_df['name'] = names.values # Re-attaching the names we saved
vis_df['cluster'] = clusters.astype(str) # The cluster from the N-dim space

# 4. Plot
fig = px.scatter(
    vis_df, 
    x='PC1', y='PC2', 
    color='cluster',
    hover_name='name',
    title="Director Style Clusters (N-Dimensional Logic Projected to 2D)"
)
fig.show()

In [28]:
temp = g.copy()
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, gen_min_span_tree=True)
clusters = clusterer.fit_predict(X)
temp['cluster'] = clusters
temp

,person_id,name,cnt,avg_f_per_fr,z_size_mv_mean,z_size_mv_g_mean,z_size_mv_std,z_size_mv_g_std,z_top1_mv_mean,z_top1_mv_g_mean,...,z_v_density_mv_g_mean,z_pct_top1_mv_mean,z_pct_top1_mv_g_mean,z_v_pct_top1_mv_mean,z_v_pct_top1_mv_g_mean,z_pct_face_mv_mean,z_pct_face_mv_g_mean,z_v_pct_face_mv_mean,z_v_pct_face_mv_g_mean,cluster
0,1,George Lucas,6,1.597648,-0.053352,0.814914,0.783111,0.766601,-0.190065,0.802465,...,0.000000,0.277344,0.991328,-0.111087,-0.397079,-0.226912,0.660440,-0.229422,0.845807,1
1,40,Orson Welles,9,1.770517,0.697528,0.110828,0.808529,0.445027,0.889053,0.204514,...,0.449870,0.888112,0.327788,0.070874,0.214835,0.630211,0.140793,1.023617,0.451762,1
2,68,Fritz Lang,33,1.873031,-0.432220,-0.640721,0.738070,0.369027,-0.447708,-0.691414,...,-0.170618,0.384600,-0.147534,0.320935,0.367024,-0.660249,-0.775029,-0.107610,-0.558840,1
3,138,Quentin Tarantino,7,1.581424,-0.217380,0.863364,0.969357,1.334367,-0.287119,0.842447,...,0.325392,0.116981,1.032773,-0.151320,-0.850129,-0.378893,0.631704,0.342752,1.289849,1
4,190,Clint Eastwood,32,2.017244,-0.416212,0.314150,0.702603,0.667530,-0.269060,0.586161,...,0.063425,-0.578839,0.328151,0.495116,0.198262,-0.136572,0.651149,-0.389156,0.490620,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328,940663,Chester M. Franklin,8,1.618405,0.397221,-0.568999,0.979921,0.185540,0.344414,-0.740273,...,0.621329,-0.074203,0.162069,0.396381,0.419476,0.359364,-0.866964,-0.004262,-0.903669,1
329,1021685,E. Mason Hopper,6,1.722700,-0.272303,-0.544496,0.875573,0.382297,-0.368806,-0.691033,...,-0.300615,-0.142999,-0.409118,0.188546,0.391993,-0.542793,-0.771631,-0.404528,-0.799606,1
330,1036754,Reginald Barker,7,1.812715,0.462617,-0.635704,0.387553,0.375400,0.511501,-0.746425,...,0.539909,-0.135964,-0.138941,0.476722,0.568228,0.635482,-0.770480,0.400587,-0.831591,1
331,1065336,Melville W. Brown,6,1.830765,-0.722426,-0.799145,0.455469,0.263664,-0.791800,-0.919072,...,-0.239791,-0.170180,-0.596750,0.235600,0.280355,-0.923024,-0.982991,-0.877001,-1.104247,1


In [30]:
# 1. Reduce your N-dimensional Z-scores to 2D
pca = PCA(n_components=2)
components = pca.fit_transform(X) # X contains your all your Z-score features

# 2. Build the visualization dataframe
vis_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
vis_df['name'] = names.values
vis_df['cluster'] = clusters.astype(str) # HDBSCAN clusters as strings

# 3. Handle the "Noise" label (-1) for the legend
vis_df['cluster'] = vis_df['cluster'].replace('-1', 'Outlier/Noise')

In [34]:
fig = px.scatter(
    vis_df, 
    x='PC1', y='PC2',
    color='cluster',
    hover_name='name',
    # template='plotly_white',
    title="HDBSCAN Style Clusters (Density-Based Projection)",
    # Optional: Discrete color sequence to make outliers stand out (e.g., grey)
    color_discrete_sequence=px.colors.qualitative.Safe 
)
# Professional touch: Force the 'Outlier' group to be a neutral grey
fig.for_each_trace(lambda t: t.update(marker_color='lightgrey') if t.name == 'Outlier/Noise' else ())

fig.show()

In [33]:
from sklearn.manifold import TSNE

# 1. Denoise with PCA first (reduces to ~5-10 dimensions)
pca_pre = PCA(n_components=min(5, X.shape[1]))
X_pca = pca_pre.fit_transform(X)

# 2. Run t-SNE
# Perplexity is the most important hyperparameter (try 5 to 30)
# It's roughly the number of neighbors each point "looks" at.
tsne = TSNE(n_components=2, perplexity=15, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_pca)

# 3. Build the Plotly DF
tsne_df = pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2'])
tsne_df['name'] = names.values
tsne_df['cluster'] = clusters.astype(str) # Keeping your HDBSCAN labels

fig = px.scatter(
    tsne_df, x='tsne_1', y='tsne_2',
    color='cluster', hover_name='name',
    title="t-SNE Projection: Local Stylistic Neighborhoods"
)
fig.show()